In [ ]:
#多项式拟合基线并求面积
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from openpyxl import Workbook
import os

def generate_baseline_for_csv(
    input_file,
    output_excel="baseline_output.xlsx",
    output_dir="baseline_figs",
    left_end=200,
    right_end=800,
    poly_order=3,
    n_rows=None
):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    df = pd.read_csv(input_file, header=None)
    
    try:
        float(df.iloc[0, 0])
    except ValueError:
        df = df.iloc[1:]

    try:
        float(df.iloc[0, 0])
    except ValueError:
        df = df.iloc[:, 1:]
    
    if n_rows is not None:
        df = df.iloc[:n_rows]
        
    n_groups, n_points = df.shape
    x = np.arange(n_points)

    wb = Workbook()
    ws = wb.active
    header = ["X"]
    for g in range(n_groups):
        header += [f"G{g+1}_Raw", f"G{g+1}_Baseline"]
    ws.append(header)

    for i in range(n_points):
        row = [int(x[i])]
        for g in range(n_groups):
            val = pd.to_numeric(df.iloc[g, i], errors='coerce')
            row += [float(val) if not np.isnan(val) else 0.0, None]
        ws.append(row)

    baselines = []
    areas = []

    for g in range(n_groups):
        y = pd.to_numeric(df.iloc[g].values, errors='coerce').astype(float)
        y = np.nan_to_num(y)
        
        mask = (x < left_end) | (x > right_end)
        coef = np.polyfit(x[mask], y[mask], poly_order)
        baseline = np.polyval(coef, x)
        baselines.append(baseline)

        area = np.trapz(baseline - y, x)
        areas.append(area)

        col_idx = 1 + (g * 2 + 2)
        for i in range(n_points):
            ws.cell(row=i + 2, column=col_idx).value = float(baseline[i])

        plt.figure(figsize=(10, 5))
        plt.plot(x, y, label="Raw Curve")
        plt.plot(x, baseline, label="Baseline")
        plt.fill_between(x, y, baseline, alpha=0.3)
        plt.title(f"Group {g+1} Raw + Baseline\nArea = {area:.1f}")
        plt.xlabel("X")
        plt.ylabel("Y")
        plt.legend()

        fig_path = os.path.join(output_dir, f"group_{g+1}_baseline.png")
        plt.savefig(fig_path, dpi=300)
        plt.close()

    wb.save(output_excel)

    plt.figure(figsize=(8, 4))
    plt.bar([f"G{i+1}" for i in range(n_groups)], areas)
    plt.title("Peak Area Comparison")
    plt.ylabel("Area")
    plt.xlabel("Group")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "area_bar_chart.png"), dpi=300)
    plt.close()

# 执行
generate_baseline_for_csv(
    input_file="1.27(2).csv",
    output_excel="1.27(2)_processed.xlsx",
    output_dir="baseline_figs",
    left_end=300,
    right_end=900,
    poly_order=3,
    n_rows=6
)